# Audio QC Visualization (qc_results/*.json)

This notebook loads QC JSON results and visualizes key metrics.
Thresholds are heuristic guidelines; tune them for your dataset.


In [ ]:
!uv add --dev matplotlib pandas seaborn plotly

In [3]:
from pathlib import Path
import json
import numpy as np

try:
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt
    import plotly.express as px
    from IPython.display import display, Markdown
except ImportError as e:
    raise SystemExit("Missing deps. Run: pip install pandas matplotlib seaborn plotly") from e

In [5]:
# Read directly from CSV (skip qc_results JSON for now)
csv_path = Path("./data/csv/noise_report_16000_with_class.csv")
if not csv_path.exists():
    csv_path = Path("./data/csv/noise_report.csv")

if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path}")
    
df = pd.read_csv(csv_path)
    
# Normalize column names (trim spaces / BOM issues)
df.columns = [str(c).strip() for c in df.columns]

def _norm_col(c):
    return ''.join(ch for ch in str(c).lower() if ch.isalnum())

# Auto-rename columns that match result_mask by normalization
target = _norm_col('result_mask')
if 'result_mask' not in df.columns:
    for c in list(df.columns):
        if _norm_col(c) == target:
            df = df.rename(columns={c: 'result_mask'})
            break

# Normalize result_mask into a readable label (Success / Missing / Other)
if "result_mask" in df.columns:
    df["result_mask"] = df["result_mask"].fillna("Unknown").astype(str)

    def _normalize_mask(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return "Unknown"
        if isinstance(v, bool):
            return "Success" if v else "Missing"
        if isinstance(v, (int, float)):
            if v == 1:
                return "Success"
            if v == 0:
                return "Missing"
            return str(v)
        s = str(v).strip().lower()
        if s in {"success", "pass", "ok", "true", "1"}:
            return "Success"
        if s in {"missing", "fail", "failed", "error", "0"}:
            return "Missing"
        return str(v)

    df["result_mask_label"] = df["result_mask"].apply(_normalize_mask)

# Convert numeric-like columns to numeric, keep metadata as strings
non_numeric = {"file_name", "file_path", "format", "subtype", "flags", "result_mask", "result_mask_label"}
for col in df.columns:
    if col not in non_numeric:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df


,avg_speech_segment_s_any,avg_speech_segment_s_ch0,avg_speech_segment_s_ch1,channel_corr_01,channels,clipping_pct_ch0,clipping_pct_ch1,clipping_pct_max,crest_db_mean,crosstalk_db_01,...,spectral_flatness_speech,spectral_rolloff95_hz_noise,spectral_rolloff95_hz_speech,speech_dropout_ratio_proxy,speech_ratio_any,speech_ratio_ch0,speech_ratio_ch1,subtype,zero_pct_max,result_mask_label
0,8.828315,7.766667,2.381400,0.000575,2,0.000007,0.018502,0.018502,20.015107,-0.003677,...,0.000733,2092.458215,1612.162401,0.213282,0.886231,0.713669,0.249316,PCM_16,6.449878,No Card
1,2.911111,4.147692,1.514118,-0.000040,2,0.000053,0.000211,0.000211,23.368056,0.000000,...,0.000639,2055.904150,1488.878752,0.317907,0.657454,0.452417,0.213235,PCM_16,8.863569,Success Overmask
2,10.096586,9.506552,2.313757,0.000430,2,0.000066,0.000021,0.000066,23.997433,-1.444893,...,0.001008,2807.011760,1609.269002,0.177440,0.918924,0.764842,0.219912,PCM_16,8.616549,Success Overmask
3,6.136165,7.337423,1.046793,0.000348,2,0.000060,0.000000,0.000060,23.552029,0.000000,...,0.001550,3018.849382,1661.788259,0.151729,0.855107,0.761657,0.137840,PCM_16,13.029186,Missing Mask
4,9.961667,12.110594,1.088743,0.000220,2,0.000096,0.000015,0.000096,26.207298,0.000000,...,0.001694,3113.518535,1683.117449,0.118024,0.919141,0.845679,0.114942,PCM_16,15.435387,Success Overmask
5,4.637892,6.620421,1.278991,0.000013,2,0.000045,0.000008,0.000045,24.834352,-0.013344,...,0.001041,3245.112538,1534.097129,0.147787,0.777334,0.664487,0.129191,PCM_16,13.792005,Success Overmask
6,6.658465,8.825695,0.797179,0.000348,2,0.000012,0.000024,0.000024,26.321975,-4.834095,...,0.000775,2465.405628,1628.413015,0.059435,0.862281,0.815675,0.096342,PCM_16,13.560552,Fail Overmask
7,5.475244,6.678938,1.624717,-0.000229,2,0.000046,0.000081,0.000081,23.682387,0.000000,...,0.001208,2970.540945,1594.458866,0.171799,0.825769,0.694750,0.156310,PCM_16,13.155515,Success Overmask
8,6.676473,9.525135,1.247114,0.000018,2,0.000016,0.000486,0.000486,24.257101,-3.493645,...,0.000666,2477.975309,1664.974502,0.113273,0.824786,0.723595,0.125968,PCM_16,10.630696,Success Mask
9,5.855081,7.588992,1.123496,-0.000386,2,0.000044,0.000000,0.000044,26.294449,0.000000,...,0.001534,3146.237132,1693.378341,0.124244,0.844175,0.763858,0.105361,PCM_16,15.844912,Success Overmask


## Data coverage check (จำนวนข้อมูลที่ใช้จริงในกราฟ)
กราฟไม่ได้จำกัดจำนวนแถว แต่จะใช้เฉพาะค่าที่เป็นตัวเลขและไม่ใช่ NaN


In [6]:
display(Markdown(f'**Total rows:** {len(df)}'))
check_cols = ["speech_ratio_any", "est_snr_db_best", "clipping_pct_max", "longest_zero_run_ms_max", "lufs_i"]
for c in check_cols:
    if c in df.columns:
        non_null = df[c].notna().sum()
        unique = df[c].nunique(dropna=True)
        display(Markdown(f'- **{c}**: non‑null={non_null}, unique={unique}'))
    else:
        display(Markdown(f'- **{c}**: column not found'))


**Total rows:** 47

- **speech_ratio_any**: non‑null=47, unique=47

- **est_snr_db_best**: non‑null=47, unique=47

- **clipping_pct_max**: non‑null=47, unique=44

- **longest_zero_run_ms_max**: non‑null=47, unique=1

- **lufs_i**: non‑null=47, unique=47

## Result mask summary (Success / Missing / Other)
ดูจำนวนไฟล์แต่ละสถานะ และตารางไฟล์ที่ไม่ผ่านเพื่อดูสาเหตุ (flags/metrics)


In [7]:
if "result_mask" in df.columns:
    display(Markdown('**result_mask (raw) counts:**'))
    display(df["result_mask"].value_counts(dropna=False))
    if "result_mask_label" in df.columns:
        display(Markdown('**result_mask_label (normalized) counts:**'))
        display(df["result_mask_label"].value_counts(dropna=False))

    def _is_success(v):
        s = str(v).lower()
        return "success" in s

    cols = [c for c in ["file_name", "result_mask", "result_mask_label", "flags", "speech_ratio_any", "est_snr_db_best", "lufs_i", "clipping_pct_max", "longest_zero_run_ms_max"] if c in df.columns]
    df_fail = df[~df["result_mask"].apply(_is_success)][cols]
    display(df_fail)
else:
    display(Markdown('**result_mask ยังไม่มี** — ชื่อคอลัมน์ในไฟล์อาจมีช่องว่าง/ตัวพิมพ์ต่างกัน'))
    display(Markdown('**Available columns:** ' + ', '.join(df.columns)))


**result_mask (raw) counts:**

result_mask
Success Overmask    19
Missing Mask         9
No Card              8
Fail Overmask        6
Success Partial      4
Success Mask         1
Name: count, dtype: int64

**result_mask_label (normalized) counts:**

result_mask_label
Success Overmask    19
Missing Mask         9
No Card              8
Fail Overmask        6
Success Partial      4
Success Mask         1
Name: count, dtype: int64

,file_name,result_mask,result_mask_label,flags,speech_ratio_any,est_snr_db_best,lufs_i,clipping_pct_max,longest_zero_run_ms_max
0,129976381_18835841.wav,No Card,No Card,NaN,0.886231,52.491720,-30.991402,0.018502,0.0625
3,130006857_18866317.wav,Missing Mask,Missing Mask,NaN,0.855107,49.342516,-34.668901,0.000060,0.0625
6,130026851_18886311.wav,Fail Overmask,Fail Overmask,NaN,0.862281,38.621876,-35.815310,0.000024,0.0625
10,130497176_18857983.wav,Fail Overmask,Fail Overmask,NaN,0.862642,50.573990,-35.747982,0.000023,0.0625
11,130510668_18871475.wav,Fail Overmask,Fail Overmask,NaN,0.903040,50.046162,-34.570407,0.003017,0.0625
12,130552803_18913610.wav,Fail Overmask,Fail Overmask,NaN,0.906611,49.454205,-34.722710,0.000155,0.0625
13,201487918_18847855.wav,Missing Mask,Missing Mask,NaN,0.850576,50.775291,-34.658419,0.000080,0.0625
16,201523985_18883922.wav,Missing Mask,Missing Mask,NaN,0.837460,51.573294,-36.115971,0.001505,0.0625
18,201537053_18896990.wav,No Card,No Card,NaN,0.842158,51.579117,-36.502491,0.000024,0.0625
20,218896568_18754925.wav,No Card,No Card,NaN,0.579178,56.503397,-31.944612,0.001587,0.0625


## File label and color mapping
Assign F1, F2, ... to each file and use consistent colors in plots.


In [8]:
def rgb_to_hex(c):
    return "#%02x%02x%02x" % (int(c[0] * 255), int(c[1] * 255), int(c[2] * 255))

if "file_name" in df.columns:
    df = df.copy()
    df["file_label"] = [f"F{i+1}" for i in range(len(df))]
    colors = sns.color_palette("tab10", n_colors=len(df))
    color_map = dict(zip(df["file_label"], colors))
    mapping_df = pd.DataFrame({
        "file_label": df["file_label"],
        "file_name": df["file_name"],
        "color": [rgb_to_hex(color_map[l]) for l in df["file_label"]],
    })
    mapping_df
else:
    color_map = None
    mapping_df = None
    None


## Guideline thresholds (heuristic)

- speech_ratio_any >= 0.15 (enough speech vs silence)
- est_snr_db_best >= 10 dB (SNR)
- clipping_pct_max <= 0.10 (% near full-scale)
- longest_zero_run_ms_max <= 500 ms (dropouts)
- lufs_i between -40 and -12 (too quiet / too loud)


In [9]:
GUIDELINES = {
    "speech_ratio_any": {"min": 0.15},
    "est_snr_db_best": {"min": 10.0},
    "clipping_pct_max": {"max": 0.10},
    "longest_zero_run_ms_max": {"max": 500.0},
    "lufs_i": {"min": -40.0, "max": -12.0},
}

def eval_range(series, min_val=None, max_val=None):
    ok = pd.Series(True, index=series.index)
    if min_val is not None:
        ok &= series >= min_val
    if max_val is not None:
        ok &= series <= max_val
    return np.where(series.isna(), "unknown", np.where(ok, "ok", "fail"))

summary_cols = []
if "file_name" in df:
    summary_cols.append("file_name")
if "file_label" in df:
    summary_cols.append("file_label")
summary = pd.DataFrame({c: df[c] for c in summary_cols})

for key, rule in GUIDELINES.items():
    if key in df:
        summary[key] = df[key]
        summary[f"{key}_ok"] = eval_range(df[key], rule.get("min"), rule.get("max"))

ok_cols = [c for c in summary.columns if c.endswith("_ok")]

def overall_state(row):
    if (row == "fail").any():
        return "fail"
    if (row == "unknown").any():
        return "unknown"
    return "ok"

if ok_cols:
    summary["guideline_overall"] = summary[ok_cols].apply(overall_state, axis=1)

summary


,file_name,file_label,speech_ratio_any,speech_ratio_any_ok,est_snr_db_best,est_snr_db_best_ok,clipping_pct_max,clipping_pct_max_ok,longest_zero_run_ms_max,longest_zero_run_ms_max_ok,lufs_i,lufs_i_ok,guideline_overall
0,129976381_18835841.wav,F1,0.886231,ok,52.491720,ok,0.018502,ok,0.0625,ok,-30.991402,ok,ok
1,129995664_18855124.wav,F2,0.657454,ok,50.421652,ok,0.000211,ok,0.0625,ok,-35.751372,ok,ok
2,130002781_18862241.wav,F3,0.918924,ok,50.477127,ok,0.000066,ok,0.0625,ok,-35.279375,ok,ok
3,130006857_18866317.wav,F4,0.855107,ok,49.342516,ok,0.000060,ok,0.0625,ok,-34.668901,ok,ok
4,130008677_18868137.wav,F5,0.919141,ok,49.945917,ok,0.000096,ok,0.0625,ok,-34.514821,ok,ok
5,130023309_18882769.wav,F6,0.777334,ok,49.564562,ok,0.000045,ok,0.0625,ok,-33.428746,ok,ok
6,130026851_18886311.wav,F7,0.862281,ok,38.621876,ok,0.000024,ok,0.0625,ok,-35.815310,ok,ok
7,130034939_18894399.wav,F8,0.825769,ok,48.448124,ok,0.000081,ok,0.0625,ok,-35.318293,ok,ok
8,130049049_18908509.wav,F9,0.824786,ok,51.320621,ok,0.000486,ok,0.0625,ok,-32.589935,ok,ok
9,130057108_18916568.wav,F10,0.844175,ok,49.479185,ok,0.000044,ok,0.0625,ok,-34.878195,ok,ok


## Flags from QC
The script also produces a "flags" column. Use it as a quick triage list.


In [10]:
cols = [c for c in ["file_name", "file_label", "flags"] if c in df]
df[cols]


,file_name,file_label,flags
0,129976381_18835841.wav,F1,NaN
1,129995664_18855124.wav,F2,NaN
2,130002781_18862241.wav,F3,NaN
3,130006857_18866317.wav,F4,NaN
4,130008677_18868137.wav,F5,NaN
5,130023309_18882769.wav,F6,NaN
6,130026851_18886311.wav,F7,NaN
7,130034939_18894399.wav,F8,NaN
8,130049049_18908509.wav,F9,NaN
9,130057108_18916568.wav,F10,NaN


## Distributions


### How to read these histograms
- X-axis is the metric value; Y-axis is count. Each color (F1/F2/...) is a file.
- The dashed lines are guideline thresholds. Points of mass left/right of a line indicate potential issues.
- Look for outliers (a bar far away from the main cluster) and skewed distributions.
- With only a few files, patterns are weak; use this mainly to spot extreme values.
- If KDE is disabled, it means too few/constant values; rely on bars only.


In [11]:
def _status_color_map(series):
    base = {
        'success': '#2ca02c',
        'missing': '#d62728',
        'fail': '#ff7f0e',
        'no card': '#9467bd',
        'unknown': '#7f7f7f',
    }
    colors = {}
    palette = px.colors.qualitative.Set2 + px.colors.qualitative.Set3
    idx = 0
    for label in sorted(series.dropna().unique()):
        s = str(label).strip().lower()
        if 'missing' in s:
            colors[label] = base['missing']
        elif 'fail' in s:
            colors[label] = base['fail']
        elif 'no card' in s or 'nocard' in s:
            colors[label] = base['no card']
        elif 'success' in s or 'pass' in s or 'ok' in s:
            colors[label] = base['success']
        elif s in ('unknown', 'nan', 'none'):
            colors[label] = base['unknown']
        else:
            colors[label] = palette[idx % len(palette)]
            idx += 1
    return colors

def _get_color_config():
    if 'result_mask' in df.columns:
        return 'result_mask', _status_color_map(df['result_mask'])
    if 'result_mask_label' in df.columns:
        return 'result_mask_label', _status_color_map(df['result_mask_label'])
    if 'file_label' in df.columns and 'mapping_df' in globals() and mapping_df is not None:
        return 'file_label', dict(zip(mapping_df['file_label'], mapping_df['color']))
    return None, None

def hist_with_guides(col, title, min_val=None, max_val=None):
    if col not in df:
        return
    col_data = df[col].dropna()
    if col_data.empty:
        if 'display' in globals():
            display(Markdown(f'**{col}**: ไม่มีข้อมูลตัวเลข (NaN ทั้งหมด)'))
        return
    color_col, color_map = _get_color_config()
    if color_col:
        fig = px.histogram(
            df,
            x=col,
            color=color_col,
            color_discrete_map=color_map,
            opacity=0.6,
            nbins=20,
            barmode='overlay',
            title=title,
        )
    else:
        fig = px.histogram(df, x=col, nbins=20, title=title)
    if min_val is not None:
        fig.add_vline(x=min_val, line_dash='dash', line_color='red')
    if max_val is not None:
        fig.add_vline(x=max_val, line_dash='dash', line_color='orange')
    fig.update_layout(bargap=0.05)
    fig.show()

HIST_EXPLAIN_TH = {
    "speech_ratio_any": "อ่านค่า speech_ratio_any: ยิ่งใกล้ 1 ยิ่งมีช่วงพูดเยอะ ถ้ากองค่าต่ำกว่าเส้น 0.15 แปลว่าไฟล์เงียบ/พูดน้อย อาจไม่เหมาะกับ ASR",
    "est_snr_db_best": "อ่านค่า SNR: ยิ่งสูงเสียงพูดชัดกว่า noise ถ้าส่วนใหญ่ต่ำกว่า 10 dB ให้พิจารณา denoise หรือคัดไฟล์",
    "clipping_pct_max": "อ่านค่า clipping: ค่ายิ่งสูงยิ่งเสียงแตก ถ้าเกิน 0.10% บ่อย ๆ คือมี clipping ชัด",
    "longest_zero_run_ms_max": "อ่านค่า dropout: ช่วงเงียบยาวผิดปกติ ถ้าเกิน 500 ms บ่อย ๆ อาจมีเสียงขาดหรือ dead air",
    "lufs_i": "อ่านค่า LUFS: ช่วงเหมาะสมอยู่ระหว่าง -40 ถึง -12 ถ้าต่ำกว่า = เบาเกินไป สูงกว่า = ดังเกินไป",
}

plot_specs = [
    ("speech_ratio_any", 0.15, None),
    ("est_snr_db_best", 10.0, None),
    ("clipping_pct_max", None, 0.10),
    ("longest_zero_run_ms_max", None, 500.0),
    ("lufs_i", -40.0, -12.0),
]

for key, vmin, vmax in plot_specs:
    hist_with_guides(key, key, vmin, vmax)
    if 'display' in globals():
        text = HIST_EXPLAIN_TH.get(key, '')
        if text:
            display(Markdown(f'**คำอธิบาย (ภาษาไทย):** {text}'))


**คำอธิบาย (ภาษาไทย):** อ่านค่า speech_ratio_any: ยิ่งใกล้ 1 ยิ่งมีช่วงพูดเยอะ ถ้ากองค่าต่ำกว่าเส้น 0.15 แปลว่าไฟล์เงียบ/พูดน้อย อาจไม่เหมาะกับ ASR

**คำอธิบาย (ภาษาไทย):** อ่านค่า SNR: ยิ่งสูงเสียงพูดชัดกว่า noise ถ้าส่วนใหญ่ต่ำกว่า 10 dB ให้พิจารณา denoise หรือคัดไฟล์

**คำอธิบาย (ภาษาไทย):** อ่านค่า clipping: ค่ายิ่งสูงยิ่งเสียงแตก ถ้าเกิน 0.10% บ่อย ๆ คือมี clipping ชัด

**คำอธิบาย (ภาษาไทย):** อ่านค่า dropout: ช่วงเงียบยาวผิดปกติ ถ้าเกิน 500 ms บ่อย ๆ อาจมีเสียงขาดหรือ dead air

**คำอธิบาย (ภาษาไทย):** อ่านค่า LUFS: ช่วงเหมาะสมอยู่ระหว่าง -40 ถึง -12 ถ้าต่ำกว่า = เบาเกินไป สูงกว่า = ดังเกินไป

## Scatter plots


### How to read these scatter plots
- Each dot is one file; axes are two metrics. Color matches the file label (F1/F2/...)
- Dots far from the main cluster are outliers to review.
- If dots form a slope, it suggests a relationship (e.g., louder files may show better SNR).
- Use guideline lines to see if points fall into acceptable regions.
- Jitter is added in some plots to separate overlapping points.


In [12]:
def _status_color_map(series):
    base = {
        'success': '#2ca02c',
        'missing': '#d62728',
        'fail': '#ff7f0e',
        'no card': '#9467bd',
        'unknown': '#7f7f7f',
    }
    colors = {}
    palette = px.colors.qualitative.Set2 + px.colors.qualitative.Set3
    idx = 0
    for label in sorted(series.dropna().unique()):
        s = str(label).strip().lower()
        if 'missing' in s:
            colors[label] = base['missing']
        elif 'fail' in s:
            colors[label] = base['fail']
        elif 'no card' in s or 'nocard' in s:
            colors[label] = base['no card']
        elif 'success' in s or 'pass' in s or 'ok' in s:
            colors[label] = base['success']
        elif s in ('unknown', 'nan', 'none'):
            colors[label] = base['unknown']
        else:
            colors[label] = palette[idx % len(palette)]
            idx += 1
    return colors

def _get_color_config():
    if 'result_mask' in df.columns:
        return 'result_mask', _status_color_map(df['result_mask'])
    if 'result_mask_label' in df.columns:
        return 'result_mask_label', _status_color_map(df['result_mask_label'])
    if 'file_label' in df.columns and 'mapping_df' in globals() and mapping_df is not None:
        return 'file_label', dict(zip(mapping_df['file_label'], mapping_df['color']))
    return None, None

SCATTER_EXPLAIN_TH = {
    "SNR vs LUFS": "อ่านกราฟ SNR vs LUFS: จุดที่อยู่ในช่วง LUFS [-40, -12] และ SNR >= 10 dB คือกลุ่มที่เสียงค่อนข้างดี (ดังพอดีและชัด) จุดนอกกรอบคือไฟล์ที่ควรตรวจ",
    "Duration vs Speech Ratio": "อ่านกราฟ Duration vs Speech Ratio: ไฟล์ยาวแต่ speech_ratio ต่ำแปลว่ามีช่วงเงียบเยอะ/hold/dead air สูง จุดที่ speech_ratio สูงและ duration พอเหมาะมักใช้ได้ดี",
}

def scatter_plot(x, y, title, x_min=None, x_max=None, y_min=None, y_max=None):
    color_col, color_map = _get_color_config()
    hover_cols = [c for c in ["file_name", "flags", "result_mask", "result_mask_label", "sample_rate", "channels", "duration_sec"] if c in df.columns]
    if color_col:
        fig = px.scatter(
            df,
            x=x,
            y=y,
            color=color_col,
            color_discrete_map=color_map,
            hover_data=hover_cols,
            title=title,
        )
    else:
        fig = px.scatter(df, x=x, y=y, hover_data=hover_cols, title=title)
    if x_min is not None:
        fig.add_vline(x=x_min, line_dash='dash', line_color='red')
    if x_max is not None:
        fig.add_vline(x=x_max, line_dash='dash', line_color='orange')
    if y_min is not None:
        fig.add_hline(y=y_min, line_dash='dash', line_color='red')
    if y_max is not None:
        fig.add_hline(y=y_max, line_dash='dash', line_color='orange')
    fig.show()

if {"lufs_i", "est_snr_db_best"}.issubset(df.columns):
    scatter_plot(
        x="lufs_i",
        y="est_snr_db_best",
        title="SNR vs LUFS",
        x_min=-40.0,
        x_max=-12.0,
        y_min=10.0,
    )
    if 'display' in globals():
        display(Markdown(f"**คำอธิบาย (ภาษาไทย):** {SCATTER_EXPLAIN_TH.get('SNR vs LUFS', '')}"))

if {"speech_ratio_any", "duration_sec"}.issubset(df.columns):
    scatter_plot(
        x="speech_ratio_any",
        y="duration_sec",
        title="Duration vs Speech Ratio",
        x_min=0.15,
    )
    if 'display' in globals():
        display(Markdown(f"**คำอธิบาย (ภาษาไทย):** {SCATTER_EXPLAIN_TH.get('Duration vs Speech Ratio', '')}"))


**คำอธิบาย (ภาษาไทย):** อ่านกราฟ SNR vs LUFS: จุดที่อยู่ในช่วง LUFS [-40, -12] และ SNR >= 10 dB คือกลุ่มที่เสียงค่อนข้างดี (ดังพอดีและชัด) จุดนอกกรอบคือไฟล์ที่ควรตรวจ

**คำอธิบาย (ภาษาไทย):** อ่านกราฟ Duration vs Speech Ratio: ไฟล์ยาวแต่ speech_ratio ต่ำแปลว่ามีช่วงเงียบเยอะ/hold/dead air สูง จุดที่ speech_ratio สูงและ duration พอเหมาะมักใช้ได้ดี

## Interpretation notes
- Values outside guidelines are not automatically bad; treat them as candidates to review.
- Tune thresholds by inspecting distributions on 100-500 files (percentile-based cutoffs work well).
- If LUFS is NaN, scipy might be missing or the audio is too short.
